# Does an Inverted Yield Curve Reliably Predict Recessions?

**Aidan Braccia — ECON 630, Assignment 6**

Question, data sources, and method are laid out in `Plan.md`. This notebook
follows that plan stage by stage.

## Stage 1 — Data pull, validation, and cleaning

Everything is pulled live from FRED on each run, so no CSVs are stored in the
repo. The validation asserts run immediately after the pull, before any
analysis, so a bad API response fails loudly instead of quietly corrupting
downstream results.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred

load_dotenv()                                   # reads FRED_API_KEY from .env (gitignored)
assert os.getenv("FRED_API_KEY"), "FRED_API_KEY not found - is .env present?"

fred = Fred(api_key=os.getenv("FRED_API_KEY"))  # key is never typed into the notebook

### Pull the three series

Each `get_series` call returns a plain `pd.Series` with a `DatetimeIndex` and no
column name — one value per month.

In [ ]:
gs10  = fred.get_series("GS10")   # 10-Year Treasury Constant Maturity Rate, monthly, percent
gs2   = fred.get_series("GS2")    # 2-Year  Treasury Constant Maturity Rate, monthly, percent
usrec = fred.get_series("USREC")  # NBER recession indicator, monthly, 1 = recession

for name, s in [("GS10", gs10), ("GS2", gs2), ("USREC", usrec)]:
    print(f"{name:6} {len(s):5} observations   {s.index.min():%Y-%m} to {s.index.max():%Y-%m}")

### Validation checks on the raw pull

These are the data-validation unit tests from `Plan.md`. They catch a scrambled
or malformed API response before anything is derived from it — if `USREC` ever
came back with a value other than 0 or 1, every recession count downstream would
be silently wrong.

In [ ]:
for name, s in [("GS10", gs10), ("GS2", gs2), ("USREC", usrec)]:
    assert s.index.is_unique, f"{name}: duplicate dates in the index"
    assert s.index.is_monotonic_increasing, f"{name}: dates are not in ascending order"

assert usrec.dropna().isin([0, 1]).all(), "USREC contains values other than 0/1"

print("raw-pull checks passed")

### Trim to the sample period and build the spread

Sample starts 1990-01, covering four NBER recessions (1990–91, 2001, 2007–09,
2020) plus the 2022–24 inversion episode.

`.loc[START:]` keeps every month from January 1990 onward — it is a slice from
that date to the end of the series, not a match on that single date.

The sign is the thing to get right: `spread = GS10 - GS2`, so an inversion is
`spread < 0`. Flipping it would silently invert the entire story.

In [ ]:
START = "1990-01-01"
gs10  = gs10.loc[START:]
gs2   = gs2.loc[START:]
usrec = usrec.loc[START:]

spread = gs10 - gs2   # 10y minus 2y, so inversion is spread < 0

print(f"sample: {spread.index.min():%Y-%m} to {spread.index.max():%Y-%m}  ({len(spread)} months)")

### Validation checks after cleaning

The alignment asserts confirm no rows were lost when the series were combined.
The spot checks confirm the sign and units are right by testing the spread in
months when the curve is known to have been inverted.

**One assert differs from `Plan.md`.** The plan spot-checked
`spread.loc["2007-09"] < 0`, but that month is **+0.51**. The inversion ahead of
the 2007–09 recession ran from 2006-02 to 2007-05, so by September 2007 the Fed
had begun cutting and the curve had re-steepened. The check moves to **2006-11**
(−0.14), the deepest month of that same inversion — same test, corrected date.

A second spot check on **2023-07** (−0.93) is added for the 2022–24 episode.

In [ ]:
assert gs10.index.equals(gs2.index), "GS10 and GS2 indexes do not match"
assert len(spread) == len(usrec),    "spread and USREC differ in length - rows lost in alignment"
assert spread.index.equals(usrec.index), "spread and USREC are not aligned date-for-date"

expected = pd.date_range(spread.index.min(), spread.index.max(), freq="MS")
assert spread.index.equals(expected), "gaps or extra dates in the monthly index"

assert spread.notna().all(), "spread has missing values - check for gaps in GS10/GS2"

assert spread.loc["2006-11"].item() < 0, \
    "expected the curve to be inverted in 2006-11, ahead of the 2007-2009 recession"
assert spread.loc["2023-07"].item() < 0, \
    "expected the curve to be inverted in 2023-07, deepest month of the 2022-24 episode"

assert spread.between(-5, 5).all(), "spread outside plausible range - check units"

print("cleaning checks passed")
print(f"spread range: {spread.min():.2f} to {spread.max():.2f} percentage points")

---

*Stage 1 reviewed and logged in `NOTES.md` on 2026-09-28. The corrected spot-check
date was written back into `Plan.md`.*

## Stage 2 — Charts

Three charts, as laid out in `Plan.md`:

1. the spread over time with recession shading,
2. each inversion episode against the recession that followed it,
3. the lead times side by side.

All three share one set of colors so they read as a single system: blue for the
series, red reserved for inversion, neutral gray for recession context and
chrome. Blue and red are a diverging pair — they encode the polarity of the
spread (above or below zero), not two arbitrary categories.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SURFACE = "#fcfcfb"   # chart surface
INK     = "#0b0b0b"   # primary text
INK2    = "#52514e"   # secondary text
MUTED   = "#898781"   # axis labels
GRID    = "#e1e0d9"   # hairline gridlines
AXIS    = "#c3c2b7"   # baseline, recession bands
BLUE    = "#2a78d6"   # the spread series
RED     = "#e34948"   # inversion


def style(ax):
    """Shared chart chrome: recessive grid, no top/right spines, muted ticks."""
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_lw(0.8)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)


def source(fig):
    fig.text(0.005, -0.02,
             "Source: FRED (GS10, GS2, USREC). Monthly, 1990-01 to "
             f"{spread.index[-1]:%Y-%m}.",
             color=MUTED, fontsize=8, ha="left", va="top")

### Defining an inversion episode

The spread crosses zero more often than there are distinct inversions — the
2006–07 episode dips below zero, pops barely positive for a month or two, and
dips again. Treating each crossing as its own episode would invent inversions
that nobody would describe as separate.

**Judgment call, not in `Plan.md`:** runs of `spread < 0` separated by a gap of
**three months or less** are merged into one episode, and single-month dips are
kept rather than filtered out. That second choice matters — both 1990-03 and
1998-06 are one-month dips, and dropping them would discard one real signal and
the clearest false alarm in the sample. `MAX_GAP` is set as a named constant so
the sensitivity of the results to this choice can be checked.

Lead time is measured from the **first** month of an episode to the first month
of the next recession.

In [ ]:
MAX_GAP = 3   # months of non-inversion tolerated inside a single episode

inverted = spread < 0
run_id   = (inverted != inverted.shift()).cumsum()
runs     = [(g.index[0], g.index[-1]) for _, g in inverted[inverted].groupby(run_id[inverted])]

episodes = []
for start, end in runs:
    if episodes:
        prev_end = episodes[-1][1]
        gap = (start.year - prev_end.year) * 12 + (start.month - prev_end.month) - 1
        if gap <= MAX_GAP:
            episodes[-1] = (episodes[-1][0], end)
            continue
    episodes.append((start, end))

# recession onsets: the month USREC flips 0 -> 1
onsets = list(usrec.index[(usrec == 1) & (usrec.shift(1) == 0)])


def months_between(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)


LAST = spread.index[-1]
rows = []
for start, end in episodes:
    following = [d for d in onsets if d > start]
    rows.append({
        "start": start,
        "end":   end,
        "recession": following[0] if following else pd.NaT,
        "lead":  months_between(start, following[0] if following else LAST),
        "min_spread": spread.loc[start:end].min(),
    })
ep = pd.DataFrame(rows)

assert len(ep) > 0, "no inversion episodes found - check the sign of spread"
assert (ep["lead"] > 0).all(), "a recession appears to precede its own inversion onset"

print(ep.to_string(index=False))

### Chart 1 — The spread over time

The fill is split at zero: blue above, red below. A single series needs no
legend, so the legend identifies only the two shaded encodings.

Note the 2020 recession band is **two months wide** and easy to mistake for a
gridline at this scale — chart 2 makes that episode legible.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4), dpi=130, facecolor=SURFACE)
style(ax)

ax.fill_between(usrec.index, 0, 1, where=usrec == 1, transform=ax.get_xaxis_transform(),
                color=AXIS, alpha=.55, lw=0, zorder=1)
ax.fill_between(spread.index, spread, 0, where=spread >= 0, color=BLUE, alpha=.16, lw=0, zorder=2)
ax.fill_between(spread.index, spread, 0, where=spread < 0,  color=RED,  alpha=.30, lw=0, zorder=2)
ax.axhline(0, color=INK2, lw=1, zorder=3)
ax.plot(spread.index, spread, color=BLUE, lw=2, zorder=4)

ax.set_ylabel("Percentage points", color=INK2, fontsize=9.5)
ax.set_title("The 10-year minus 2-year Treasury spread, 1990–2026",
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015, "Below zero the yield curve is inverted. Shaded bands are NBER recessions.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
ax.legend(handles=[Patch(facecolor=AXIS, alpha=.55, label="NBER recession"),
                   Patch(facecolor=RED, alpha=.30, label="Inverted (spread < 0)")],
          loc="upper right", frameon=False, fontsize=9, labelcolor=INK2)
source(fig)
plt.show()

### Chart 2 — Each episode against the recession that followed

One row per episode, on a calendar axis. The red bar spans the inverted months;
the arrow runs from inversion onset to the next recession onset. This is the
chart that shows the two awkward cases: the 2020 recession arrived with no
inversion before it, and the 2022 inversion has yet to be followed by one.

In [ ]:
# contiguous recession spans, for shading
rec_id = (usrec != usrec.shift()).cumsum()
rec_spans = [(b.index[0], b.index[-1] + pd.DateOffset(months=1))
             for _, b in usrec[usrec == 1].groupby(rec_id[usrec == 1])]

fig, ax = plt.subplots(figsize=(11, 4.2), dpi=130, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for s0, e0 in rec_spans:
    ax.axvspan(s0, e0, color=AXIS, alpha=.55, lw=0, zorder=1)
ax.grid(axis="x", color=GRID, lw=.8, zorder=0)
ax.set_axisbelow(True)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color(AXIS); ax.spines["bottom"].set_lw(.8)
ax.tick_params(colors=MUTED, labelsize=9, length=0)

y_rows = list(range(len(ep)))[::-1]          # earliest episode on top
for i, (_, r) in zip(y_rows, ep.iterrows()):
    pending = pd.isna(r.recession)
    target  = LAST if pending else r.recession
    ax.barh(i, (r.end + pd.DateOffset(months=1)) - r.start, left=r.start,
            height=.34, color=RED, alpha=.85, lw=0, zorder=3)
    ax.annotate("", xy=(target, i + .40), xytext=(r.start, i + .40),
                arrowprops=dict(arrowstyle="->", color=RED if pending else INK2, lw=1.2,
                                ls=(0, (4, 2)) if pending else "-", shrinkA=0, shrinkB=0),
                zorder=4)
    ax.text(r.start + (target - r.start) / 2, i + .50,
            f"{r.lead}+ mo — no recession" if pending else f"{r.lead} mo",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
            color=RED if pending else INK, zorder=5)

ax.set_yticks(y_rows)
ax.set_yticklabels([f"{r.start:%b %Y}" for _, r in ep.iterrows()], color=INK2, fontsize=10)
ax.set_ylim(-.7, len(ep) - .25)
ax.set_xlim(pd.Timestamp("1989-01-01"), pd.Timestamp("2027-06-01"))
ax.set_title("Each inversion episode and the recession that followed",
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015,
        "Bars span the inverted months; arrows run from inversion onset to the next recession onset.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
ax.legend(handles=[Patch(facecolor=RED, alpha=.85, label="Inversion episode"),
                   Patch(facecolor=AXIS, alpha=.55, label="NBER recession")],
          loc="lower left", frameon=False, fontsize=9, labelcolor=INK2, ncol=2)
ax.annotate("2020 recession —\nno inversion preceded it",
            xy=(pd.Timestamp("2020-02-01"), 1.65), xytext=(pd.Timestamp("2013-06-01"), 2.15),
            color=INK2, fontsize=9, ha="center",
            arrowprops=dict(arrowstyle="->", color=MUTED, lw=1))
source(fig)
plt.show()

### Chart 3 — Lead times side by side

The plan called this the chart that decides the question: a consistent signal
produces bars of similar length, a noisy one produces scatter. The 2022 episode
is drawn open and hatched rather than filled, because its bar is a *censored*
observation — the recession hasn't happened, so 49 months is a lower bound, not
a lead time. It carries its own label so the distinction never rests on the
hatching alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.0), dpi=130, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.grid(axis="x", color=GRID, lw=0.8)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(AXIS); ax.spines["bottom"].set_lw(0.8)
ax.tick_params(colors=MUTED, labelsize=9, length=0)

y_rows = list(range(len(ep)))[::-1]
for i, (_, r) in zip(y_rows, ep.iterrows()):
    pending = pd.isna(r.recession)
    ax.barh(i, r.lead, height=.58, zorder=2,
            color="none" if pending else BLUE,
            edgecolor=RED if pending else "none",
            lw=1.6, hatch="///" if pending else None)
    ax.text(r.lead + 1.2, i,
            f"{r.lead}+ months, no recession yet" if pending else f"{r.lead} months",
            va="center", ha="left", fontsize=9.5, fontweight="bold",
            color=RED if pending else INK)

ax.set_yticks(y_rows)
ax.set_yticklabels([f"{r.start:%b %Y}" for _, r in ep.iterrows()], color=INK2, fontsize=10)
ax.set_xlim(0, 72)
ax.set_xlabel("Months from inversion onset", color=INK2, fontsize=9.5)
ax.set_title("Lead times are not consistent", color=INK, fontsize=13,
             fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015, "Months from the first month of each inversion to the next NBER recession.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
source(fig)
plt.show()

---

*Stage 2 reviewed and logged in `NOTES.md` on 2026-09-28. The `MAX_GAP = 3`
episode definition was accepted and written into `Plan.md`.*

## Stage 3 — Statistical tests

Two tests from `Plan.md`: a two-sample t-test on the spread twelve months
earlier, and a logistic regression of recession status on that same lagged
spread.

First the remaining cleaning step — `inverted` and `spread_lag12`.

In [ ]:
inverted     = spread < 0        # boolean: was the curve inverted this month?
spread_lag12 = spread.shift(12)  # the spread as it stood 12 months before each date

# shift(12) must line up PAST spread with TODAY's recession status, not the reverse.
assert spread_lag12.loc["2001-01"].item() == spread.loc["2000-01"].item(), \
    "spread_lag12 is not looking backwards - check the sign of the shift"
assert spread_lag12.iloc[:12].isna().all(), "first 12 months should be NaN after a 12-month lag"

print(f"inverted months: {inverted.sum()} of {len(inverted)}")
print(f"spread_lag12 spans {spread_lag12.first_valid_index():%Y-%m} to "
      f"{spread_lag12.index[-1]:%Y-%m}")

### Test 1 — Two-sample t-test

Compare the spread twelve months earlier in months that turned out to be
recession months against months that did not. Welch's version (`equal_var=False`)
is used because the two groups differ enormously in size and there is no reason
to assume equal variances.

In [ ]:
from scipy import stats

data = pd.DataFrame({"usrec": usrec, "spread_lag12": spread_lag12}).dropna()

rec_months = data.loc[data["usrec"] == 1, "spread_lag12"]
non_months = data.loc[data["usrec"] == 0, "spread_lag12"]

print(f"observations: {len(data)}   recession months: {int(data['usrec'].sum())}"
      f"  ({data['usrec'].mean():.1%} of the sample)\n")
print(f"mean spread 12 months earlier, recession months    : {rec_months.mean():+.3f}  (n={len(rec_months)})")
print(f"mean spread 12 months earlier, non-recession months: {non_months.mean():+.3f}  (n={len(non_months)})")
print(f"difference                                         : {rec_months.mean() - non_months.mean():+.3f} pp\n")

t_stat, p_val = stats.ttest_ind(rec_months, non_months, equal_var=False)
print(f"Welch t = {t_stat:.3f},  p = {p_val:.2e}")

The sign is what the yield-curve story predicts: the spread a year before a
recession month averages **0.34** percentage points, against **1.07** in
non-recession months — about three-quarters of a point flatter.

**But that p-value is not worth much, and it is worth being precise about why.**
The t-test treats all 428 months as independent draws. They are nothing of the
sort: the 31 recession months are four contiguous episodes, and both the spread
and recession status are strongly autocorrelated from one month to the next.
The effective sample size is closer to four than to 428, so the test overstates
its confidence by a wide margin. The *difference in means* is a fair
description; the p-value beside it is not.

### Test 2 — Logistic regression

`USREC ~ spread_lag12`, one lag only. This goes past what has been lectured —
`statsmodels` appeared in the toolkit overview but was not taught — so each
piece is spelled out, and it is flagged as self-directed in the writeup.

- `sm.add_constant` adds the intercept column. Without it the model is forced
  through zero, which silently changes what the coefficient means.
- `Logit(y, X).fit()` estimates by maximum likelihood.
- The coefficient is on the **log-odds** scale, so `np.exp(coef)` converts it to
  an odds ratio per one-percentage-point change in the spread.

In [ ]:
import numpy as np
import statsmodels.api as sm

y = data["usrec"]
X = sm.add_constant(data[["spread_lag12"]])   # intercept + one regressor

logit = sm.Logit(y, X).fit(disp=0)
print(logit.summary())

coef = logit.params["spread_lag12"]
print(f"\nodds ratio per +1pp of spread: {np.exp(coef):.3f}")
print(f"i.e. each extra percentage point of spread 12 months ago multiplies the")
print(f"odds of being in recession by about {np.exp(coef):.2f}.")

### Test 2b — The same model with autocorrelation-robust standard errors

Not in `Plan.md`, added because the plan's own stated limitation demands it. The
table above reports `Covariance Type: nonrobust`, which assumes independent
observations — the same assumption the t-test broke. Re-fitting with
Newey-West (HAC) standard errors at a 12-month bandwidth keeps the coefficient
identical and asks only how much of its apparent precision survives.

In [ ]:
logit_hac = sm.Logit(y, X).fit(disp=0, cov_type="HAC", cov_kwds={"maxlags": 12})

comparison = pd.DataFrame({
    "coefficient": [logit.params["spread_lag12"], logit_hac.params["spread_lag12"]],
    "std error":   [logit.bse["spread_lag12"],    logit_hac.bse["spread_lag12"]],
    "z":           [logit.tvalues["spread_lag12"], logit_hac.tvalues["spread_lag12"]],
    "p-value":     [logit.pvalues["spread_lag12"], logit_hac.pvalues["spread_lag12"]],
}, index=["nonrobust", "HAC (12 lags)"]).round(4)

print(comparison.to_string())
print(f"\nstandard error grows {logit_hac.bse['spread_lag12'] / logit.bse['spread_lag12']:.1f}x "
      "once autocorrelation is allowed for.")

The coefficient does not move — the point estimate is what it is. The standard
error more than doubles, and the p-value goes from 0.0001 to **0.09**. Under the
conventional 5% threshold the relationship stops being significant.

This is the single most important number in the notebook, and it is the one a
naive reading would have missed entirely. The `nonrobust` table is not wrong
arithmetic; it is answering a question nobody should ask of four recessions.

---

## Conclusion

**The question:** does a negative 10-year minus 2-year spread reliably precede
U.S. recessions, or is it noisier than its reputation suggests?

**On this evidence, noisier — which is what `Plan.md` expected going in.**

The direction of the relationship is real and consistent. The spread a year
before a recession month is about three-quarters of a percentage point flatter
than otherwise, the logit coefficient is negative as theory predicts, and four
of the five inversion episodes since 1990 were followed by a recession. Nothing
here says the signal is meaningless.

What it is not is *reliable*, in the sense of supporting a forecast:

1. **Lead times do not cluster.** 5, 34, 14, and 23 months. Knowing an
   inversion has begun tells you remarkably little about when.
2. **It produces false alarms.** The one-month dip in June 1998 was followed by
   34 quiet months, and by the time the 2001 recession arrived a separate,
   deeper inversion had been running for over a year.
3. **It misses recessions.** The curve was positive throughout the two years
   before the 2020 recession. A signal that cannot see a pandemic coming is a
   signal with a bounded domain.
4. **Its most recent firing has not resolved.** The 2022–24 inversion is the
   deepest in the sample at −0.93 and, 49 months on, has not been followed by a
   recession.
5. **The statistical significance does not survive an honest standard error.**
   p = 0.0001 becomes p = 0.09 once autocorrelation is allowed for.

**The limitation `Plan.md` flagged up front turned out to be the finding.** With
four recessions in thirty-six years, there was never enough independent
variation to support a confident causal claim, and the gap between the naive and
HAC standard errors is that shortage made visible. The charts and the difference
in means describe the pattern well. The p-values should not be read as
confirming it.

**What would have changed the conclusion:** lead times falling in a narrow band,
no unexplained episodes, and a coefficient that stayed significant under robust
standard errors. None of the three holds.

---

**Stop here.** Per `NOTES.md`, review before the writeup: confirm `add_constant`
is present (it is — see the `const` row), that the `shift(12)` direction assert
passes, and decide whether you accept adding the HAC comparison, which is not in
`Plan.md` and is the reason the conclusion is hedged the way it is.